# Перекрёстные комбинации моделей v1 и v2

- **A:** классификатор v1 (ConvNeXt V2 Tiny, 512) + YOLO v2.1 (YOLO26l, 640)
- **B:** классификатор v2 (ConvNeXt V2 Base, 640) + YOLO v1 (YOLO26m, 512)

Каждый снимок ресайзится отдельно под классификатор и под YOLO, боксы возвращаются в исходные 1024 по масштабу YOLO. Пороги подбираются в координатах 1024 по исходной разметке `stage_2_train_labels.csv`.

> **Оценка не совсем честная.** v1 и v2 обучались на разных сплитах, поэтому часть снимков val одна из моделей видела при обучении, и пороги подбираются на частично знакомых снимках. Оценка Kaggle при этом честная — закрытый тест не видела ни одна модель; неточные пороги могут её только занизить.

## 0. Установка, Drive, данные

In [1]:
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg ultralytics timm

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
from pathlib import Path
import pandas as pd

project = Path("/content/drive/MyDrive/rsna_project_final")
zip_path = Path("/content/drive/MyDrive/rsna-pneumonia-detection-challenge.zip")
extract_dir = Path("/content/extract_dir")
if not (extract_dir / ".done").exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    (extract_dir / ".done").touch()
train_image_path = extract_dir / "stage_2_train_images"

meta = pd.read_csv(project / "meta_classifier_ConvNeXt_V2_Base_with_aug.csv")   # val v2 — для порогов
labels_1024 = pd.read_csv(extract_dir / "stage_2_train_labels.csv").dropna()
gt_1024 = {pid: g[["x", "y", "width", "height"]].to_numpy() for pid, g in labels_1024.groupby("patientId")}

## 1. Препроцессинг и метрика соревнования

In [4]:
import sys
import numpy as np
import cv2
import pydicom

def percentile_clip(data, lower_percentile=0.5, upper_percentile=99.5):
    low_val, high_val = np.percentile(data, [lower_percentile, upper_percentile])
    return np.clip(data, low_val, high_val)

def read_dicom(dicom_path):
    dcm = pydicom.dcmread(dicom_path)
    img = dcm.pixel_array.astype("float32")
    if dcm.PhotometricInterpretation == "MONOCHROME1":
        img = img.max() - img
    img = percentile_clip(img)
    img = (img - np.min(img)) / (np.max(img) - np.min(img) + sys.float_info.epsilon)
    return np.round(img * 255).astype(np.uint8)

def resize_img(img, size=512):
    resized_img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return resized_img, resized_img.shape[0] / img.shape[0]

In [5]:
THRESHOLDS = np.round(np.arange(0.4, 0.76, 0.05), 2)   # 0.4, 0.45, ..., 0.75

def iou_matrix(a, b):
    """IoU каждого бокса из a с каждым из b. Боксы — (n, 4): x, y, width, height."""
    ax1, ay1, ax2, ay2 = a[:, 0], a[:, 1], a[:, 0] + a[:, 2], a[:, 1] + a[:, 3]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]
    iw = np.clip(np.minimum(ax2[:, None], bx2) - np.maximum(ax1[:, None], bx1), 0, None)
    ih = np.clip(np.minimum(ay2[:, None], by2) - np.maximum(ay1[:, None], by1), 0, None)
    inter = iw * ih
    union = (a[:, 2] * a[:, 3])[:, None] + b[:, 2] * b[:, 3] - inter
    return inter / union


def image_ap(gt, pred, scores):
    """AP одного снимка по правилам Kaggle. None — снимок не учитывается."""
    if len(gt) == 0 and len(pred) == 0:
        return None
    if len(gt) == 0 or len(pred) == 0:
        return 0.0
    pred = pred[np.argsort(-scores)] # сначала самые уверенные
    ious = iou_matrix(pred, gt) # (число предсказаний, число настоящих)
    aps = []
    for t in THRESHOLDS:
        matched, tp = set(), 0
        for i in range(len(pred)):
            cand = [j for j in range(len(gt)) if j not in matched and ious[i, j] > t]
            if cand: # совпал с ещё не занятым настоящим боксом
                matched.add(max(cand, key=lambda j: ious[i, j]))
                tp += 1
        fp, fn = len(pred) - tp, len(gt) - tp
        aps.append(tp / (tp + fp + fn))
    return float(np.mean(aps))

## 2. Функция проверки комбинации

Прогон val через обе модели, подбор порогов `t2 × conf`, отправка на Kaggle.

In [6]:
import timm, torch
from ultralytics import YOLO
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def run_combo(name, cls_name, cls_size, cls_weights, yolo_weights, yolo_size):
    cls = timm.create_model(cls_name, pretrained=False, num_classes=3, in_chans=1)
    cls.load_state_dict(torch.load(cls_weights, map_location=DEVICE))
    cls = cls.to(DEVICE).eval()
    yolo = YOLO(str(yolo_weights))

    def run_models(path, conf=0.01):
        """p2 классификатора и боксы YOLO [x, y, w, h] в координатах исходного снимка."""
        img = read_dicom(path)
        c_img, _ = resize_img(img, cls_size)
        y_img, scale = resize_img(img, yolo_size)
        x = torch.from_numpy(c_img).float().div(255)[None, None]
        with torch.no_grad():
            p2 = torch.softmax(cls(((x - 0.449) / 0.226).to(DEVICE)), dim=1)[0, 2].item()
        r = yolo.predict(np.repeat(y_img[..., None], 3, axis=2), imgsz=yolo_size, conf=conf, verbose=False)[0]
        xyxy = r.boxes.xyxy.cpu().numpy() / scale
        return p2, np.c_[xyxy[:, :2], xyxy[:, 2:] - xyxy[:, :2]], r.boxes.conf.cpu().numpy()

    # --- подбор порогов на val ---
    val_ids = meta.loc[meta["split"] == "val", "patientId"]
    val_out = {pid: run_models(train_image_path / f"{pid}.dcm") for pid in tqdm(val_ids, desc="val")}

    def score(t2, conf):
        aps = []
        for pid, (p2, pred, s) in val_out.items():
            keep = (s >= conf) & (p2 >= t2)
            ap = image_ap(gt_1024.get(pid, np.zeros((0, 4))), pred[keep], s[keep])
            if ap is not None:
                aps.append(ap)
        return np.mean(aps)

    score_val, t2, conf = max((score(t, c), t, c)
                              for t in np.arange(0, 0.81, 0.05) for c in np.arange(0.05, 0.81, 0.05))
    print(f"{name}: prob2 >= {t2:.2f}, conf >= {conf:.2f} | val {score_val:.4f} (не совсем честно)")

    # --- Kaggle ---
    rows = []
    for path in tqdm(sorted((extract_dir / "stage_2_test_images").glob("*.dcm")), desc="kaggle"):
        p2, pred, s = run_models(path, conf=conf)
        pred_str = "" if p2 < t2 else " ".join(
            f"{c:.3f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}" for (x, y, w, h), c in zip(pred, s))
        rows.append(dict(patientId=path.stem, PredictionString=pred_str))
    sub = pd.DataFrame(rows)
    sub.to_csv(project / f"submission_{name}.csv", index=False)
    print("снимков с боксами:", (sub["PredictionString"] != "").sum(), "из", len(sub))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## 3. Комбинация A: классификатор v1 (Tiny, 512) + YOLO v2.1 (26l, 640)

In [8]:
run_combo(
    name="A_tiny512_yolo26l640",
    cls_name="convnextv2_tiny.fcmae_ft_in22k_in1k", cls_size=512,
    cls_weights=project / "classifier_ConvNeXt_V2_Tiny_no_aug" / "classifier_best_for_f1_classifier_ConvNeXt_V2_Tiny_no_aug.pt",
    yolo_weights=project / "yolo" / "split_v2_yolo26l_2" / "weights" / "best.pt", yolo_size=640,
)

val: 100%|██████████| 5337/5337 [07:48<00:00, 11.38it/s]


A_tiny512_yolo26l640: prob2 >= 0.40, conf >= 0.25 | val 0.2139 (не совсем честно)


kaggle: 100%|██████████| 3000/3000 [04:20<00:00, 11.50it/s]


снимков с боксами: 687 из 3000


**Результат A:** t2 = 0.40 · conf = 0.25 · val 0.214 · Kaggle Private **0.117** · Public 0.042

## 4. Комбинация B: классификатор v2 (Base, 640) + YOLO v1 (26m, 512)

In [10]:
run_combo(
    name="B_base640_yolo26m512",
    cls_name="convnextv2_base.fcmae_ft_in22k_in1k_384", cls_size=640,
    cls_weights=project / "classifier_ConvNeXt_V2_Base_with_aug" / "classifier_best_for_f1_ConvNeXt_V2_Base_with_aug.pt",
    yolo_weights=project / "yolo" / "split_v1_yolo26m" / "weights" / "best.pt", yolo_size=512,
)

val: 100%|██████████| 5337/5337 [08:01<00:00, 11.09it/s]


B_base640_yolo26m512: prob2 >= 0.20, conf >= 0.30 | val 0.2065 (не совсем честно)


kaggle: 100%|██████████| 3000/3000 [04:28<00:00, 11.16it/s]


снимков с боксами: 703 из 3000


**Результат B:** t2 = 0.20 · conf = 0.30 · val 0.207 · Kaggle Private **0.116** · Public 0.056